In [1]:
import tqdm
import torch
from load_dotenv import load_dotenv
import os
import json
import re

# Basic transformer libraries
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from transformers import Mistral3ForConditionalGeneration, FineGrainedFP8Config

# Tools
import requests
from collections import defaultdict
from typing import Callable, Any
from dataclasses import dataclass

c:\Users\dubos\Documents\Development\Portfolio\Growth_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Settings informations
Once the code is finished, the following cell should be the only one needing modification to interact with it.

In [2]:
model_id = "mistralai/Ministral-3-3B-Instruct-2512"

prompt_system = "You are a helpful librarian trying to recommand books based on a query by a user. " \
"Your answer includes 3 books, with a short summary of each, their type and the link to download the book. " \
"If no link is available, simply say link: Not found. Do not suggest other buying/downloading platforms"

prompt_user = "Can you recommend an adventuring book to me please ?"




## Loading .env data

In [3]:
load_dotenv()
GUT_HOST = os.getenv("GUT_HOST")

In [4]:
GUT_HOST

'https://project-gutenberg-books-api1.p.rapidapi.com'

## Model initiation

In [6]:
# model_id = "mistralai/Ministral-3-3B-Instruct-2512"
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=FineGrainedFP8Config(dequantize=True)
)

# model = AutoModelForCausalLM.from_pretrained(model)
tokenizer = AutoTokenizer.from_pretrained(model_id, device_map="auto", dtype=torch.bfloat16)
   


Loading weights: 100%|██████████| 458/458 [00:06<00:00, 76.28it/s]
[transformers] The tokenizer you are loading from 'mistralai/Ministral-3-3B-Instruct-2512' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


## Tool Creation

In [7]:
def fetch_book_url(title: str):
    """
    Get the url of a given book.

    Args:
        title: The title of the book that is being looked up. 
    """
    endpoint = "/books"
    headers = {
        "X-RapidAPI-Key" : os.getenv("RAPIDAPI"),
        "X-RapidAPI-Host": "project-gutenberg-free-books-api1.p.rapidapi.com"
        }
    params = {
        "title" : title,

    }
    response = requests.get(GUT_HOST+endpoint, params=params, headers=headers)
    if response.status_code == 200:
        response = response.json()
        if (response.get("results") and len(response["results"]) > 0):
            result = response["results"][0]["formats"]["text/plain; charset=us-ascii"]
        else:
            result = 'No url was found in the Project Gutenberg'
    return result #["results"][0]["formats"]["text/plain; charset=us-ascii"]

# tools = [fetch_book_url]

@dataclass
class Tool:
    name: str
    func: Callable
    description: str
    schema: dict

tool_registry = {
    'fetch_book_url' : Tool(
        name='fetch_book_url',
        func=fetch_book_url,
        description="""
            Get the url of a given book.
        
            Args:
                title: The title of the book that is being looked up. 
            """,
        schema={
            'type':'object',
            'properties':{
                'title':{'type': 'string'}
            },
            'required': ['title']
        }
    )
}





## Translation to mistral API

In [8]:
def tool_to_mistral_schema(tool: Tool) -> dict:
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description.strip(),
            "parameters": tool.schema
        }
    }
mistral_tools = [tool_to_mistral_schema(tool) for tool in tool_registry.values()]

## Agent Class

In [17]:
import re
import json
import inspect
from typing import Callable


class Agent:
    """
    Provider-agnostic LLM agent with tool calling support.

    Designed for local models (e.g., Mistral) but adaptable to any LLM provider.
    Separates:
    - System prompt (static, defines agent behavior)
    - Context window (dynamic, conversation history)
    - Tools (schema + implementation pairs)
    """

    def __init__(self, model, tokenizer, system_prompt: str | None = None):
        """
        Initialize the agent.

        Args:
            model: The LLM model (e.g., from transformers)
            tokenizer: The tokenizer (e.g., from transformers)
            system_prompt: Optional system prompt defining agent behavior
        """
        self.model = model
        self.tokenizer = tokenizer
        self.system_prompt = system_prompt
        self.context_window = []
        self.tools = {}  # Dict keyed by tool name for O(1) lookup

    # ============================================================
    # SYSTEM PROMPT MANAGEMENT
    # ============================================================

    def set_system_prompt(self, prompt: str):
        """Update the system prompt"""
        self.system_prompt = prompt
        print(f"✓ System prompt set ({len(prompt)} chars)")

    def get_system_prompt(self) -> str:
        """Get current system prompt"""
        return self.system_prompt or ""

    # ============================================================
    # CONTEXT WINDOW MANAGEMENT
    # ============================================================

    def add_to_context(self, role: str, content):
        """
        Add a message to the context window.

        Args:
            role: "user", "assistant", or "system"
            content: Message content (str or list of dicts for tool results)
        """
        self.context_window.append({"role": role, "content": content})

    def clear_context(self):
        """Clear the entire conversation history"""
        self.context_window = []
        print("✓ Context window cleared")

    def get_context(self) -> list[dict]:
        """Get a copy of the current context window"""
        return self.context_window.copy()

    def truncate_context(self, keep_last_n: int = 10):
        """
        Keep only the last N messages in context.
        Useful for token efficiency.
        """
        if len(self.context_window) > keep_last_n:
            self.context_window = self.context_window[-keep_last_n:]
            print(f"✓ Context truncated to last {keep_last_n} messages")

    # ============================================================
    # MESSAGE BUILDING
    # ============================================================

    def _build_messages(self) -> list[dict]:
        """
        Build complete message list for LLM.
        Structure: [system (if exists), ...context_window]
        """
        messages = []

        if self.system_prompt:
            messages.append({"role": "system", "content": self.system_prompt})

        messages.extend(self.context_window)

        return messages

    # ============================================================
    # TOOL MANAGEMENT
    # ============================================================

    @staticmethod
    def validate_schema(name: str, parameters: dict, func: Callable) -> tuple[bool, str]:
        """
        Validate that schema structure matches function signature.

        Returns:
            (is_valid, error_message)
        """
        errors = []

        # 1. Check schema structure
        if not isinstance(parameters, dict):
            errors.append("parameters must be a dict")
            return False, "; ".join(errors)

        if parameters.get("type") != "object":
            errors.append("parameters['type'] must be 'object'")

        if "properties" not in parameters or not isinstance(parameters["properties"], dict):
            errors.append("parameters must have 'properties' dict")

        if "required" not in parameters or not isinstance(parameters["required"], list):
            errors.append("parameters must have 'required' list")

        if errors:
            return False, "; ".join(errors)

        # 2. Validate required params exist in properties
        properties = parameters["properties"]
        required = parameters["required"]

        for param in required:
            if param not in properties:
                errors.append(f"Required parameter '{param}' not in properties")

        # 3. Validate function signature matches schema
        sig = inspect.signature(func)
        func_params = set(sig.parameters.keys())
        schema_params = set(properties.keys())

        missing_params = set(required) - func_params
        if missing_params:
            errors.append(
                f"Function missing required parameters: {missing_params}. "
                f"Function has: {func_params}"
            )

        extra_params = func_params - schema_params - {"self"}
        if extra_params:
            errors.append(
                f"Function has parameters not in schema: {extra_params}. "
                f"This may cause issues when LLM calls the tool."
            )

        # 4. Check all properties have a type
        for param, prop in properties.items():
            if "type" not in prop:
                errors.append(f"Property '{param}' missing 'type' field")

        if errors:
            return False, "; ".join(errors)

        return True, ""

    def register_tool(
        self,
        name: str,
        description: str,
        parameters: dict,
        func: Callable,
        strict: bool = True
    ):
        """
        Register a tool with both schema and implementation.

        Args:
            name: Tool name (must be unique)
            description: What the tool does (for LLM)
            parameters: JSON schema for parameters
            func: Python callable that executes the tool
            strict: If True, raise error on validation failure
        """
        is_valid, error_msg = self.validate_schema(name, parameters, func)

        if not is_valid:
            if strict:
                raise ValueError(f"Invalid schema for tool '{name}':\n{error_msg}")
            else:
                print(f"⚠️  Warning for tool '{name}':\n{error_msg}")

        # Store keyed by name for O(1) lookup
        self.tools[name] = {
            "type": "function",
            "function": {
                "name": name,
                "description": description,
                "parameters": parameters
            },
            "implementation": func
        }

        print(f"✓ Registered tool: {name}")

    def _get_tool_schemas(self) -> list[dict]:
        """
        Extract just the schemas (without implementations) to send to LLM.
        The LLM doesn't need or want the Python callables.
        """
        return [
            {
                "type": tool["type"],
                "function": tool["function"]
            }
            for tool in self.tools.values()
        ]

    def execute_tool(self, tool_name: str, tool_input: dict) -> str:
        """
        Execute a tool by name with given input.

        Args:
            tool_name: Name of the tool to execute
            tool_input: Dict of arguments to pass to the tool

        Returns:
            String result of tool execution
        """
        if tool_name not in self.tools:
            raise ValueError(f"Unknown tool: {tool_name}")

        tool = self.tools[tool_name]
        return str(tool["implementation"](**tool_input))

    # ============================================================
    # LLM CALLS
    # ============================================================

    def call_llm(self) -> str:
        """
        Call Mistral and get ONLY newly generated response.
        Uses TOKEN SLICING method.
        """
        messages = self._build_messages()

        tokenized_prompt = self.tokenizer.apply_chat_template(
            messages,
            tools=self._get_tool_schemas(),
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(self.model.device)

        # Remember prompt length to extract only new tokens
        prompt_length = tokenized_prompt["input_ids"].shape[1]

        # Generate
        outputs = self.model.generate(**tokenized_prompt, max_new_tokens=500)

        # Extract ONLY the new tokens
        new_tokens = outputs[0][prompt_length:]
        decoded = self.tokenizer.decode(new_tokens, skip_special_tokens=False)

        # Clean up stray tokens
        decoded = decoded.replace("</s>", "").strip()
        
        # Remove stray [/INST] markers that sometimes appear
        decoded = decoded.replace("[/INST]", "").strip()

        return decoded.strip()

    def call_llm_parse(self) -> str:
        """
        Call Mistral and get ONLY newly generated response.
        Uses STRING PARSING method - splits on [/INST].
        """
        messages = self._build_messages()

        tokenized_prompt = self.tokenizer.apply_chat_template(
            messages,
            tools=self._get_tool_schemas(),
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(self.model.device)

        # Generate
        outputs = self.model.generate(**tokenized_prompt, max_new_tokens=500)

        # Decode full response
        full_response = self.tokenizer.decode(outputs[0], skip_special_tokens=False)

        # Extract only what comes after [/INST]
        parts = full_response.split("[/INST]")

        if len(parts) > 1:
            new_response = parts[-1].strip()
            new_response = new_response.replace("</s>", "").strip()
            return new_response

        return full_response.strip()

    # ============================================================
    # TOOL CALL PARSING
    # ============================================================

    def parse_tool_calls(self, response_text: str) -> list[dict]:
        """
        Extract tool calls from Mistral's [TOOL_CALLS] format.

        Format: [TOOL_CALLS]function_name[ARGS]{"param": "value"}[TOOL_CALLS]...
        """
        tool_call_pattern = r'\[TOOL_CALLS\](.+?)\[ARGS\](.+?)(?=\[TOOL_CALLS\]|$)'
        matches = re.findall(tool_call_pattern, response_text, re.DOTALL)

        tool_calls = []
        for i, (func_name, args_json) in enumerate(matches):
            try:
                parsed_args = json.loads(args_json.strip())
            except json.JSONDecodeError as e:
                print(f"Failed to parse args for tool {func_name}: {e}")
                parsed_args = {}

            tool_calls.append({
                "id": str(i),
                "name": func_name.strip(),
                "input": parsed_args
            })

        return tool_calls

    def has_tool_calls(self, response_text: str) -> bool:
        """Check if response contains tool calls"""
        return "[TOOL_CALLS]" in response_text

    # ============================================================
    # AGENT LOOP
    # ============================================================

    def run(self, user_prompt: str, max_iterations: int = 10, use_parse: bool = False) -> str:
        """
        Run the agentic loop.

        Manages:
        - System prompt (static context)
        - Context window (conversation history)
        - Tool calls and execution

        Args:
            user_prompt: Initial user message
            max_iterations: Maximum iterations before giving up
            use_parse: If True, use string parsing LLM call. If False, use token slicing.

        Returns:
            Final text response from the agent
        """
        # Add initial user message to context
        self.add_to_context("user", user_prompt)

        # Choose which LLM call method to use
        llm_caller = self.call_llm_parse if use_parse else self.call_llm

        for iteration in range(max_iterations):
            print(f"\n{'='*60}")
            print(f"Iteration {iteration + 1}/{max_iterations}")
            print(f"{'='*60}")

            # Step 1: Call LLM
            print("Calling LLM...")
            response_text = llm_caller()
            print(f"Response (first 300 chars):\n{response_text}...\n")

            # Step 2: Check if tool calls are present
            if self.has_tool_calls(response_text):
                print("→ Tool calls detected")

                # Add assistant response to context
                self.add_to_context("assistant", response_text)

                # Step 3: Parse tool calls
                tool_calls = self.parse_tool_calls(response_text)
                print(f"→ Parsed {len(tool_calls)} tool call(s)")

                # Step 4: Execute tools
                tool_results = []
                for tool_call in tool_calls:
                    tool_name = tool_call["name"]
                    tool_input = tool_call["input"]
                    tool_id = tool_call["id"]

                    try:
                        print(f"  • Executing '{tool_name}' with {tool_input}")
                        result = self.execute_tool(tool_name, tool_input)
                        print(f"    ✓ Result: {result[:100]}...")

                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": tool_id,
                            "content": result
                        })
                    except Exception as e:
                        print(f"    ✗ Error: {e}")
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": tool_id,
                            "content": f"Error: {str(e)}",
                            "is_error": True
                        })

                # Step 5: Add tool results back to context
                self.add_to_context("tool", json.dumps(tool_results))
                print("→ Tool results added to context")

            else:
                # No tool calls = final response
                print("→ No tool calls, returning final response")
                self.add_to_context("assistant", response_text)
                return response_text

        return "Agent hit max iterations"


In [ ]:
# ============================================================
# USAGE EXAMPLE
# ============================================================

# from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
# model_name = "mistralai/Mistral-7B-Instruct-v0.2"
# model = AutoModelForCausalLM.from_pretrained(model_name)
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# # Define system prompt
# system_prompt = """You are a helpful librarian trying to recommend books based on a query by a user.
# Your answer includes 3 books, with a short summary of each, their type and the link to download the book.
# If no link is available, simply say link: Not found.
# Do not suggest other buying/downloading platforms.
# Be concise and friendly in your recommendations."""

# Create agent
agent = Agent(model, tokenizer, system_prompt=prompt_system)

# Define external tool functions
# def fetch_book_url(title: str) -> str:


# def search_books(query: str) -> str:
#     """Search for books by query"""
#     return f"Found 5 books matching '{query}'"

# Register tools
agent.register_tool(
    name="fetch_book_url",
    description="Get the URL of a given book",
    parameters={
        "type": "object",
        "properties": {
            "title": {"type": "string", "description": "Book title"}
        },
        "required": ["title"]
    },
    func=fetch_book_url
)


# Run agent
result = agent.run("Can you recommend an adventuring book to me please?")
print(f"\n{'='*60}")
print(f"FINAL RESPONSE:")
print(f"{'='*60}")
print(result)


[transformers] Both `max_new_tokens` (=500) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ Registered tool: fetch_book_url

Iteration 1/10
Calling LLM...
Response (first 300 chars):
[TOOL_CALLS]fetch_book_url[ARGS]{"title": "The Name of the Wind"}[TOOL_CALLS]fetch_book_url[ARGS]{"title": "The Lies of Locke Lamora"}[TOOL_CALLS]fetch_book_url[ARGS]{"title": "The City of Brass: A Dubai Mystery"}...

→ Tool calls detected
→ Parsed 3 tool call(s)
  • Executing 'fetch_book_url' with {'title': 'The Name of the Wind'}
    ✓ Result: No url was found in the Project Gutenberg...
  • Executing 'fetch_book_url' with {'title': 'The Lies of Locke Lamora'}
    ✓ Result: No url was found in the Project Gutenberg...
  • Executing 'fetch_book_url' with {'title': 'The City of Brass: A Dubai Mystery'}


[transformers] Both `max_new_tokens` (=500) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    ✓ Result: No url was found in the Project Gutenberg...
→ Tool results added to context

Iteration 2/10
Calling LLM...
Response (first 300 chars):
Here are three adventurous books for you:

1. **"The Name of the Wind" by Patrick Rothfuss**
   - **Type:** Fantasy Adventure
   - **Summary:** Kvothe, a prodigy with a mysterious past, recounts his extraordinary life story, filled with music, magic, and danger. This epic tale blends adventure, romance, and deep philosophical themes.

   link: Not found

2. **"The Lies of Locke Lamora" by Scott Lynch**
   - **Type:** Heist Fantasy
   - **Summary:** In a city ruled by a secretive guild of thieves, Locke Lamora and his crew pull off daring heists while navigating political intrigue and personal betrayals. This book is packed with wit, strategy, and rich world-building.

   link: Not found

3. **"The City of Brass: A Dubai Mystery" by S.A. Chakraborty**
   - **Type:** Historical Fantasy Adventure
   - **Summary:** Set in a magical 1920s Cair